In [12]:

#spark.sql("DROP TABLE IF EXISTS dim_operational_status")

#spark.sql("DROP TABLE IF EXISTS fact_wind_power")

#from pyspark.sql import SparkSession

#spark._jsparkSession.catalog().refreshTable("dim_operational_status")

#spark._jsparkSession.catalog().refreshTable("fact_wind_power")


StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 14, Finished, Available, Finished, False)

In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number




StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 15, Finished, Available, Finished, False)

In [14]:
silver_trable_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Silver.Lakehouse/Tables/dbo/wind_power"

df = spark.read.format("delta").load(silver_trable_path)



StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 16, Finished, Available, Finished, False)

In [15]:
#   Create the data dimension table

data_dim = df.select("date", "day", "month", "quarter", "year").distinct() \
    .withColumnRenamed("date","date_id")

# Create the time dimension table

time_dim = df.select("time", "hour_of_day", "mint_of_hour", "sec_of_minute", "time_period").distinct() \
    .withColumnRenamed("time","time_id")


# Create a turbine dimension table

turbine_dim = df.select("turbine_name", "capacity", "location_name", "latitude", "longitude", "region").distinct() \
    .withColumn("turbine_id", row_number().over(Window.orderBy("turbine_name", "capacity", "location_name", "latitude", "longitude", "region")))


# Create The Operation Status Dimension Table


operational_status_dim = df.select("status", "responsible_department").distinct() \
    .withColumn("status_id", row_number().over(Window.orderBy("status", "responsible_department")))

StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 17, Finished, Available, Finished, False)

In [16]:
df = df.join(turbine_dim, ["turbine_name", "capacity", "location_name", "latitude", "longitude", "region"], "left") \
        .join(operational_status_dim, ["status", "responsible_department"], "left")

StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 18, Finished, Available, Finished, False)

In [17]:
fact_table = df.select("production_id",'date','time', 'status_id', 'turbine_id', 'wind_speed', 'wind_direction', 'energy_produced' ) \
    .withColumnRenamed('date','date_id') \
    .withColumnRenamed('time', 'time_id') 

StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 19, Finished, Available, Finished, False)

In [18]:
# Paths to gold tables

gold_date_dim_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Gold.Lakehouse/Tables/dbo/dim_date"

gold_time_dim_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Gold.Lakehouse/Tables/dbo/dim_time"

gold_turbine_dim_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Gold.Lakehouse/Tables/dbo/dim_turbine"

gold_operational_status_dim_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Gold.Lakehouse/Tables/dbo/dim_operational_status"

gold_fact_dim_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Gold.Lakehouse/Tables/dbo/fact_wind_power"

StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 20, Finished, Available, Finished, False)

In [19]:

data_dim.write.mode("overwrite").format("delta").save(gold_date_dim_path)

time_dim.write.mode("overwrite").format("delta").save(gold_time_dim_path)

turbine_dim.write.mode("overwrite").format("delta").save(gold_turbine_dim_path)

operational_status_dim.write.mode("overwrite").format("delta").option("mergeSchema", "true").save(gold_operational_status_dim_path)

fact_table.write.mode("overwrite").format("delta").option("mergeSchema", "true").save(gold_fact_dim_path)

StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 21, Finished, Available, Finished, False)

In [20]:
%%sql

SELECT max(date_id)
FROM LH_Wind_Power_Gold.dbo.fact_wind_power


StatementMeta(, aaba65b2-ce20-4a08-b828-59695e06d261, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>